# Optimize orbitals

In quantum chemistry and variational quantum algorithms, the choice of single-particle orbital basis has a significant impact on the accuracy and compact representation of correlated electronic states. While canonical molecular orbitals obtained from mean-field Hartree-Fock theory are optimal for the single-determinant reference state, they are generally not optimal for correlated wave functions—such as Configuration Interaction (CISD), Multi-Configuration Self-Consistent Field (MCSCF/CASSCF), or states generated by variational ansatzes in a truncated active space.

Orbital optimization finds an [orbital rotation](../explanations/orbital-rotation.ipynb) (a unitary transformation $U \in \mathrm{U}(N)$ or orthogonal transformation $O \in \mathrm{SO}(N)$ of the $N$ spatial orbitals) that minimizes the energy expectation value for a given set of one- and two-body reduced density matrices (1- and 2-RDMs) with respect to a [MolecularHamiltonian](../api/stubs/ffsim.MolecularHamiltonian.rst#ffsim.MolecularHamiltonian).

The [ffsim.optimize_orbitals](../api/stubs/ffsim.optimize_orbitals.rst#ffsim.optimize_orbitals) function implements this optimization using JAX to compute analytical gradients with respect to the Lie algebra parametrization of the orbital rotation group, interfaced with SciPy's `minimize` optimizer (defaulting to L-BFGS-B).

This guide demonstrates how to:

- Set up a molecular Hamiltonian and active space with PySCF.
- Compute 1- and 2-RDMs for a correlated state and wrap them in [ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix).
- Find the energy-minimizing orbital rotation using `ffsim.optimize_orbitals`.
- Evaluate the optimized energy using both rotated RDMs and rotated Hamiltonians.
- Restrict to real orbital rotations vs. allow complex rotations.
- Transform molecular orbital coefficients for downstream calculations.
- Compute RDMs directly from state vectors using [ffsim.rdms](../api/stubs/ffsim.rdms.rst#ffsim.rdms).

## Build molecule and active space Hamiltonian

We'll use a nitrogen molecule ($N_2$) at a stretched bond length ($R = 2.4\text{ \AA}$) in the 6-31G basis set. At stretched bond lengths, static correlation is pronounced, making orbital relaxation effects particularly evident. We freeze the core $1s$ orbitals and construct an active space on the remaining orbitals.

In [1]:
import pyscf

import ffsim

# Build N2 molecule at stretched geometry
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, -1.2)], ["N", (0, 0, 1.2)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space by freezing 1s core orbitals
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Run RHF and obtain molecular Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
mol_hamiltonian = mol_data.hamiltonian
norb = mol_data.norb

print(f"Number of active orbitals: {norb}")
print(f"Number of active electrons: {mol_data.nelec}")

converged SCF energy = -108.508552140411
Number of active orbitals: 16
Number of active electrons: (5, 5)


## Compute correlated state and reduced density matrices

Next, we run a Configuration Interaction Singles and Doubles (CISD) calculation within the active space and extract the 1- and 2-RDMs. We wrap them in [ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix), which provides convenient methods to rotate RDMs and compute energy expectation values with molecular Hamiltonians.

The initial energy evaluated using the canonical Hartree-Fock orbitals matches the CISD total energy.

In [2]:
import pyscf.ci

# Run CISD in the active space
cisd = pyscf.ci.CISD(scf, frozen=n_frozen).run()
cisd_energy = cisd.e_tot

# Extract 1- and 2-RDMs in the active space
rdm1 = cisd.make_rdm1()[n_frozen:, n_frozen:]
rdm2 = cisd.make_rdm2()[n_frozen:, n_frozen:, n_frozen:, n_frozen:]
rdm = ffsim.ReducedDensityMatrix(rdm1, rdm2)

# Compute energy using canonical Hartree-Fock orbitals
initial_energy = rdm.expectation(mol_hamiltonian)
print(f"Initial CISD energy: {initial_energy:.8f} Ha")

E(CISD) = -108.585322102141  E_corr = -0.076769961730
Initial CISD energy: -108.58532210 Ha


## Optimize orbitals

We now call [ffsim.optimize_orbitals](../api/stubs/ffsim.optimize_orbitals.rst#ffsim.optimize_orbitals) to find the orbital rotation matrix $U$ that minimizes the energy of the RDMs with respect to `mol_hamiltonian`.

Setting `return_optimize_result=True` returns both the optimal orbital rotation matrix and the `scipy.optimize.OptimizeResult` object containing optimizer statistics such as the number of iterations and function evaluations.

Once the optimal rotation $U$ is found, the minimized energy can be computed in two equivalent ways:

1. Rotating the reduced density matrices by $U$:
   ```python
   rdm.rotated(orbital_rotation).expectation(mol_hamiltonian)
   ```
2. Rotating the Hamiltonian by $U^\dagger$:
   ```python
   rdm.expectation(mol_hamiltonian.rotated(orbital_rotation.T.conj()))
   ```

Because reduced density matrices are dual to Hamiltonian operator tensors, rotating the RDMs by $U$ gives the exact same energy as rotating the Hamiltonian by $U^\dagger$.

In [3]:
# Optimize orbitals with real rotations
orbital_rotation, result = ffsim.optimize_orbitals(
    rdm,
    mol_hamiltonian,
    return_optimize_result=True,
)

# Compute the energy in the optimized orbital basis
energy_rotated_rdm = rdm.rotated(orbital_rotation).expectation(mol_hamiltonian)
energy_rotated_ham = rdm.expectation(
    mol_hamiltonian.rotated(orbital_rotation.T.conj())
)

print(f"Optimized energy:    {energy_rotated_rdm:.8f} Ha")
print(f"Energy lowering:     {(energy_rotated_rdm - initial_energy) * 1000:.4f} mHa")
print(f"Dual rotation check: {energy_rotated_ham:.8f} Ha")
print(f"Optimizer success:   {result.success}")
print(f"Iterations:          {result.nit}")

Optimized energy:    -108.58613394 Ha
Energy lowering:     -0.8118 mHa
Dual rotation check: -108.58613394 Ha
Optimizer success:   True
Iterations:          7


## Real vs. complex orbital rotations

The `real` keyword argument controls whether the optimization is restricted to real orthogonal rotations ($O \in \mathrm{SO}(N)$) or allowed to explore general complex unitary rotations ($U \in \mathrm{U}(N)$):

- When `real=True`, the rotation is parameterized by $N(N - 1) / 2$ skew-symmetric generator parameters.
- When `real=False`, the rotation is parameterized by $N^2$ anti-Hermitian generator parameters.
- When `real=None` (the default), `ffsim.optimize_orbitals` automatically detects whether the RDMs, Hamiltonian, and initial orbital rotation are all real-valued, and defaults to real optimization if so.

If we initialize with a complex orbital rotation (or pass `real=False`), the optimizer optimizes over the full unitary group:

In [4]:
import numpy as np
import scipy.linalg

# Generate a small complex perturbation to the initial orbital rotation
rng = np.random.default_rng(241127903049053917326682280086371763733)
perturbation = 1e-4 * ffsim.random.random_antihermitian(norb, seed=rng)
initial_orbital_rotation = scipy.linalg.expm(perturbation)

# Optimize orbitals allowing complex rotations
orbital_rotation_complex, result_complex = ffsim.optimize_orbitals(
    rdm,
    mol_hamiltonian,
    initial_orbital_rotation=initial_orbital_rotation,
    return_optimize_result=True,
)

energy_complex = rdm.rotated(orbital_rotation_complex).expectation(mol_hamiltonian)

print(f"Optimized energy (complex): {energy_complex:.8f} Ha")
print(f"Number of parameters:       {len(result_complex.x)} (vs {len(result.x)} real)")
print(f"Is complex rotation matrix: {np.iscomplexobj(orbital_rotation_complex)}")

Optimized energy (complex): -108.58613394 Ha
Number of parameters:       256 (vs 120 real)
Is complex rotation matrix: True


## Update molecular orbital coefficients

To use the optimized orbitals in subsequent calculations (such as an iterative CASSCF cycle or Sample-based Quantum Diagonalization), we apply the orbital rotation matrix $U$ to the molecular orbital coefficient matrix $C$:

$$
C_{\text{opt}} = C_{\text{active}} U
$$

where $C_{\text{active}}$ contains the active-space columns of the original Hartree-Fock MO coefficients.

In [5]:
# Transform molecular orbital coefficients for downstream calculations
mo_coeff_active = scf.mo_coeff[:, active_space]
mo_coeff_optimized = mo_coeff_active @ orbital_rotation

print(f"Original active MO coefficients shape:  {mo_coeff_active.shape}")
print(f"Optimized active MO coefficients shape: {mo_coeff_optimized.shape}")

Original active MO coefficients shape:  (28, 16)
Optimized active MO coefficients shape: (28, 16)


## Computing RDMs from state vectors

In variational quantum algorithms like VQE, the quantum state is often represented as a state vector rather than density matrices obtained from an external quantum chemistry package. You can compute the 1- and 2-RDMs of an ffsim state vector directly using [ffsim.rdms](../api/stubs/ffsim.rdms.rst#ffsim.rdms).

The following example demonstrates computing RDMs from a state vector and forming a `ReducedDensityMatrix` object for orbital optimization:

In [6]:
# Construct the reference Hartree-Fock state vector
hf_vec = ffsim.hartree_fock_state(norb, mol_data.nelec)

# Compute 1- and 2-RDMs directly from the state vector
rdm1_hf, rdm2_hf = ffsim.rdms(hf_vec, norb, mol_data.nelec, rank=2)
rdm_hf = ffsim.ReducedDensityMatrix(rdm1_hf, rdm2_hf)

# Expectation value of the Hartree-Fock state equals the SCF energy
hf_energy = rdm_hf.expectation(mol_hamiltonian)
print(f"Hartree-Fock RDM energy: {hf_energy:.8f} Ha")
print(f"Hartree-Fock SCF energy: {scf.e_tot:.8f} Ha")

Hartree-Fock RDM energy: -108.50855214 Ha
Hartree-Fock SCF energy: -108.50855214 Ha
